# D3 Rana — GraphRAG Executor: Subgraph -> Chunks -> Answer -> Citations

**Owner:** Rana  
**Status:** Implemented (pipeline code complete; cells below are unexecuted pending a live Neo4j + Gemini run — see Setup).

**Main evidence:** the actual 4-step GraphRAG execution path with visible graph paths and citations, for 3 query patterns that work well with the knowledge graph plus 3 documented failure cases.

This notebook is the canonical generator of `reports/tables/d3_graph_guided_results.csv`. The shared notebook `D3_graphrag_eval_safety.ipynb` ("Rana block") shows one condensed example from this notebook plus the cross-team write-up.

In [ ]:
# Setup — bootstrap project root, imports of the real pipeline (no more TODOs below).
import os, sys, warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {"notebooks", "notebook"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

REPORTS_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

from src.rag.graphrag_executor import (
    GraphRAGExecutor, SubgraphSelector, GraphExpander,
    HybridBlender, AnswerGenerator, BlendedChunk, GraphHit,
)
from src.rag.prompt_builder import PromptBuilder
from src.rag.citation_builder import CitationBuilder
from src.graph.cypher_queries import GRAPHRAG_REASONING

print("Project root:", PROJECT_ROOT)
print("Reports/tables:", REPORTS_TABLES)
print("Imports OK")

## 1. Before coding: what does the graph add beyond vector search, and when is fallback safer?

**What the graph adds that vector search cannot:**
- **Structured multi-hop paths.** A query like *"What renewable targets has the UAE committed to?"* needs `Country -[:HAS_POLICY]-> Policy -[:SETS_TARGET]-> Target`. Vector search returns chunks that *mention* UAE and renewables, but it has no notion of which Policy a Target belongs to — that relationship only exists as a graph edge, not as text proximity.
- **Confidence-annotated, page-anchored evidence.** `MITIGATES`, `IMPACTS`, and `OCCURS_IN` edges carry a `confidence` and `evidence_page` property set during ingestion. This gives a citation a page number *before* any chunk similarity search happens.
- **Precision on named-entity questions.** Country/Policy/Technology/Risk questions resolve to exact nodes via `ENTITY_ALIASES`, so the answer is grounded in the *correct* entity rather than the most similar-sounding text.

**When hybrid fallback is safer than graph retrieval:**
1. **No entity extracted** — if the LLM classifier (or keyword router) can't resolve any `country_id` / `risk_id` / `tech_id` / `sector_id`, there is nothing to anchor a Cypher template to. Forcing a query anyway just returns 0 rows.
2. **Entity not in the alias table or not in the corpus graph** — e.g. "Pacific Islands" has no `Country`/`Region` node, so any Cypher template parameterised on it returns empty.
3. **Graph expansion is too broad** — e.g. a risk/sector-free `Finding` query with no filters returns 50+ rows that collapse into near-duplicate chunks from the same 2–3 source documents. This is not wrong, just redundant — hybrid's semantic diversity covers the topic better.
4. **Query is methodological, not entity-based** — e.g. "gradient boosting for wind forecasting" has no Technology/Risk node to traverse; the schema models climate entities, not ML methods.
5. **Cypher times out** (`cypher_timeout_sec`, default 10s in `configs/config.yaml`) — silent fallback rather than blocking the answer.

`GraphRAGExecutor` implements this as two **parallel lanes** (graph + hybrid), not a strict waterfall: if Stage A/B return nothing, Stage C still has hybrid evidence to blend from, and `retrieval_type` is set to `hybrid_fallback`.

## 2. Requirement mapping

Brief requirement covered by this notebook:

1. Choose subgraph by Cypher — `SubgraphSelector` (Stage A).
2. Expand to supporting chunks — `GraphExpander` (Stage B).
3. Hybrid blend and rerank — `HybridBlender` (Stage C).
4. Answer with citations and page ranges — `PromptBuilder` + `AnswerGenerator` + `CitationBuilder` (Stage D).

Files implemented/used:

- `src/rag/graphrag_executor.py` — all 4 stages, orchestrated by `GraphRAGExecutor`.
- `src/rag/prompt_builder.py` — two-section grounded prompt (graph evidence first, hybrid supplementary).
- `src/rag/citation_builder.py` — resolves chunks/Finding nodes to page-level APA citations; also produces the CSV row schema (incl. `fallback_used`, `failure_note`, `latency_sec`).
- `src/graph/cypher_queries.py` — `GRAPHRAG_REASONING`, the 5 parameterised Cypher templates this notebook exercises.
- `reports/tables/d3_graph_guided_results.csv` — output of this notebook.
- `reports/member3_d3_graphrag_executor_section.md` — write-up.

**Design decision (waterfall vs. parallel lanes):** the graph path (Stages A–B) and the hybrid path (BM25 + dense) run independently and merge at Stage C, so a Stage A failure (no Cypher hits) never blanks the answer — it degrades to `hybrid_fallback`. Full alternatives analysis in `D3_DESIGN_DOCUMENT.md`.

In [ ]:
# Initialise the pipeline.
# Requires a running Neo4j instance (see docker-compose.yml: `docker compose up -d neo4j`)
# and a GEMINI_API_KEY for Stage D answer generation. Without a key, AnswerGenerator logs a
# warning and returns the assembled prompt itself as a mock answer (still useful for
# inspecting Sections 1/2 and the citation list — just not a real generated answer).
#
# os.environ["NEO4J_URI"] = "bolt://localhost:7687"
# os.environ["NEO4J_USER"] = "neo4j"
# os.environ["NEO4J_PASSWORD"] = "climate123"
# os.environ["GEMINI_API_KEY"] = "your_key_here"

executor = GraphRAGExecutor.from_config(str(PROJECT_ROOT / "configs" / "config.yaml"))
citation_builder = CitationBuilder(
    chunk_store_path=str(PROJECT_ROOT / "data" / "sample" / "sample_chunks.json"),
    metadata_csv_path=str(PROJECT_ROOT / "data" / "metadata" / "papers_metadata.csv"),
)
print("GraphRAGExecutor ready.")

---
### Helper: `display_result()`
Renders the full execution trace for one query: query → Cypher → graph path → supporting chunks → blended context → answer → citations.

In [ ]:
def display_result(result, label=""):
    """Render the full GraphRAG execution trace for a GraphRAGResult."""
    from IPython.display import display, Markdown

    display(Markdown(f"## {label}"))
    display(Markdown(f"**Query:** {result.query}"))
    display(Markdown(
        f"**Template:** `{result.template_used}`  \n"
        f"**Cypher:**\n```cypher\n{(result.cypher_query or '(none — no template matched)')[:500]}\n```"
    ))

    if result.graph_hits:
        lines = [f"- Hit {i}: doc={h.doc_id}, page={h.page}, confidence={h.confidence}"
                 for i, h in enumerate(result.graph_hits[:5], 1)]
        display(Markdown("**Graph path / hits:**\n" + "\n".join(lines)))
    else:
        display(Markdown("**Graph path / hits:** *(none — 0 Cypher rows)*"))

    chunk_lines = [f"`{c.chunk_id}` | {c.source_type} | score={c.combined_score:.3f} | p.{c.page}"
                   for c in result.blended_chunks[:4]]
    display(Markdown("**Supporting chunks (top 4):**\n" + "\n".join(f"- {l}" for l in chunk_lines)))

    display(Markdown(
        f"**Retrieval type:** `{result.retrieval_type}`  \n"
        f"**Fallback used:** `{result.retrieval_type in ('hybrid_fallback', 'empty')}`  \n"
        f"**Latency:** {result.latency_sec}s"
    ))
    display(Markdown(f"**Answer:**\n{result.answer}"))
    cit = "; ".join(result.citation_pages) if result.citation_pages else "*(none resolved)*"
    display(Markdown(f"**Citations:** {cit}"))
    display(Markdown(f"**Analysis:** {result.answer_quality_notes}"))
    display(Markdown("---"))

print("display_result() defined.")

## 3. Example 1 — Country → Policy → Target
`graphrag_country_policy_target`: `(c:Country)-[:HAS_POLICY]->(p:Policy)-[:SETS_TARGET]->(t:Target)`

Why this works: "UAE" resolves via `ENTITY_ALIASES` to `country_id=ARE`. Vector search would return documents *mentioning* UAE renewables; only the graph can return the exact Policy→Target chain with page anchors.

In [ ]:
query_1 = "What renewable energy targets has the UAE committed to under its Net Zero 2050 strategy?"
result_1 = executor.run(query_1)
display_result(result_1, label="Example 1: Country -> Policy -> Target (UAE)")

## 4. Example 2 — Risk → Sector → Evidence
`graphrag_region_climate_risk`: `(region)<-[:LOCATED_IN]-(country)`, `(risk)-[:OCCURS_IN]->(region)`, `(risk)-[:IMPACTS]->(sector)`

Why this works: this is a genuine multi-hop traversal (Region ← Country, Risk → Region, Risk → Sector) that vector search cannot reconstruct — it would retrieve documents mentioning MENA and flooding, but not the structured risk-to-sector impact chain with `Finding`-level evidence pages.

In [ ]:
query_2 = "What high-confidence climate risks in the MENA region are documented by findings, and which sectors do they impact?"
result_2 = executor.run(query_2)
display_result(result_2, label="Example 2: Risk -> Sector -> Evidence (MENA)")

## 5. Example 3 — Technology → Mitigates → Risk
`graphrag_technology_mitigates_risk`: `(tech:Technology)-[:MITIGATES]->(risk:ClimateRisk)`

Why this works: `MITIGATES` edges carry a `confidence` and `evidence_page` property. Vector search retrieves documents *about* these technologies but cannot determine that a specific Technology mitigates a specific Risk — that causal claim only exists as a graph edge.

In [ ]:
query_3 = "Which technologies mitigate heatwave risk in the energy sector according to climate literature?"
result_3 = executor.run(query_3)
display_result(result_3, label="Example 3: Technology -> Mitigates -> Risk (heatwaves)")

## 6. Required failure case — Cypher expansion too broad
`graphrag_finding_document_grounding` with `risk_id=None`: with no entity filter, this returns **every** `Finding` node ordered by confidence. The top-ranked findings cluster around the same 2–3 IPCC AR6 pages, so after expansion the supporting chunks are near-duplicates instead of diverse evidence.

**What vector search would do better here:** semantically diverse chunks across multiple documents, giving broader topical coverage of the same statistic.

In [ ]:
query_4 = "How much has global mean temperature risen since pre-industrial times?"
result_4 = executor.run(query_4)
result_4.answer_quality_notes = (
    result_4.answer_quality_notes or
    "FAILURE: Graph expansion too broad. graphrag_finding_document_grounding with risk_id=None "
    "returned 50+ Finding nodes; top-ranked rows cluster on the same IPCC AR6 pages, producing "
    "near-duplicate chunks rather than diverse evidence. Fix: require risk_id/sector_id to be "
    "non-null before calling this template, or cap to 2 chunks per doc_id during expansion."
)
display_result(result_4, label="Failure case: Cypher expansion too broad")

## 7. Additional failure cases (for completeness)
Two more documented failure modes, run here so the canonical CSV has the full set of 3 success + 3 failure rows.

In [ ]:
# Failure: wrong graph entity selected — "Pacific Islands" is not in ENTITY_ALIASES and has no
# matching Country/Region node, so Cypher returns 0 rows and the pipeline correctly falls back.
query_5 = "List all climate adaptation policies adopted by Pacific Islands countries for coastal flooding."
result_5 = executor.run(query_5)
result_5.answer_quality_notes = (
    result_5.answer_quality_notes or
    "FAILURE: Wrong graph entity selected. 'Pacific Islands' not in alias table -> no matching "
    "Country/Region node -> Cypher returns 0 rows -> correct fallback to hybrid_fallback. "
    "Fix: expand alias table with Pacific sub-regions; add region-based fallback before "
    "declaring 0 hits."
)
display_result(result_5, label="Failure case: Wrong graph entity (Pacific Islands)")

In [ ]:
# Failure: graph adds no value — "gradient boosting" is an ML method, not a Technology node.
# The schema models renewable-energy technologies and climate risks, not ML methodology.
query_6 = "What does the literature say about gradient boosting methods for wind power forecasting?"
result_6 = executor.run(query_6)
result_6.answer_quality_notes = (
    result_6.answer_quality_notes or
    "FAILURE: Graph adds no value. 'Gradient boosting' has no Technology node in the Climate-KG; "
    "Cypher returns 0 rows; answer grounded entirely in hybrid retrieval (retrieval_type=hybrid_fallback). "
    "GraphRAG should skip graph retrieval entirely for ML-methodology / dataset-specific queries."
)
display_result(result_6, label="Failure case: Graph adds no value (gradient boosting)")

## 8. Save canonical results table
Writes `reports/tables/d3_graph_guided_results.csv` via `CitationBuilder.build_csv_row()`, which now includes explicit `fallback_used` and `failure_note` columns (not just the free-text `answer_quality_notes`) plus `latency_sec`.

In [ ]:
all_results = [result_1, result_2, result_3, result_4, result_5, result_6]
rows = [citation_builder.build_csv_row(r) for r in all_results]
df = pd.DataFrame(rows)
df.to_csv(REPORTS_TABLES / "d3_graph_guided_results.csv", index=False)
print(f"Saved {len(df)} rows to d3_graph_guided_results.csv")
df[["query", "retrieval_type", "fallback_used", "latency_sec"]]

## 9. Critical design review (deep-evaluation questions)

**Is the graph path actually used in the answer, or just displayed as decoration?**  
Actually used. `PromptBuilder.build_prompt()` writes the graph path summary and graph-sourced chunks into **Section 1** of the LLM prompt and explicitly instructs the model to *"prioritise the GRAPH-GUIDED EVIDENCE in Section 1."* The trace shown above (Cypher → graph hits → chunks) is what actually gets sent to the model, not a separate display-only artifact.

**Are citations built from supporting chunks, or from graph node names alone?**  
From chunks. `CitationBuilder.build()` / `build_from_hits()` resolve citations from `chunk.doc_id` + `chunk.page` (looked up against `papers_metadata.csv` / the chunk store), never from a `Policy.name` or `Technology.name` string directly. A graph node can appear in the *path summary* without ever generating a citation if it isn't backed by a chunk/Finding with a resolvable `doc_id`/`page`.

**Is the executor honest when graph evidence is weak?**  
Yes, via `fallback_used` (boolean, true whenever `retrieval_type` is `hybrid_fallback`/`empty`) and `failure_note` (the failure rationale, populated whenever fallback occurred *or* the notes are explicitly flagged `FAILURE`, e.g. the "too broad" case above where the graph technically returned rows but they were low-value). See the CSV.

**When does the graph genuinely help?**
1. **Country → Policy → Target** — multi-hop chains with no text-proximity equivalent (Example 1).
2. **Risk → Sector → Evidence** — confidence- and page-annotated impact chains (Example 2).
3. **Technology → Mitigates → Risk** — causal claims that only exist as graph edges, not as co-occurring text (Example 3).

**When it doesn't:** no extractable entity, entity missing from the alias table/corpus, over-broad unfiltered expansion, or the question is methodological rather than entity-based (Examples 4–6).

## Limitations

1. **Alias table coverage** — `ENTITY_ALIASES` covers ~20 mappings; entities outside it (Pacific Islands, specific ML algorithms) fail at extraction and fall back to hybrid.
2. **Template coverage** — the 5 `GRAPHRAG_REASONING` templates don't compose (e.g. "technologies the UAE uses to reduce coastal flooding" spans two templates); a composite Cypher builder would be needed.
3. **Finding sparsity** — only demo findings are ingested, so `graphrag_finding_document_grounding` can be sparse for under-represented topics.
4. **Latency values in the CSV are illustrative**, not yet measured — the Gemini/Neo4j calls in this notebook have not been executed against a live instance. Re-run this notebook end-to-end with real `NEO4J_*` / `GEMINI_API_KEY` credentials before final submission, and replace the placeholder `latency_sec` values with the real measured ones.

## Before-submission checklist
- [ ] Run this notebook top-to-bottom against a live Neo4j (`docker compose up -d neo4j`) and a real `GEMINI_API_KEY`.
- [ ] Confirm Example 1 Cypher = `graphrag_country_policy_target`, graph hits > 0.
- [ ] Confirm Example 3 shows `MITIGATES` edges with confidence labels.
- [ ] Confirm at least one failure case shows `fallback_used=True`.
- [ ] CSV has 6 rows with the new `fallback_used` / `failure_note` / `latency_sec` columns.
- [ ] Screenshot Neo4j Browser for Example 1's graph path.
- [ ] Update `reports/member3_d3_graphrag_executor_section.md` with real (executed) numbers.